# Evolutionary escape race — does the death wave clear before resistance sweeps? (RUNG-19)

**CPU only — no GPU, ~1 GB RAM.** A stochastic lattice race: the recognition-gated death wave (RUNG-13) vs a tumour that keeps dividing and mutating to RESISTANT during treatment.

Output: P(cure) vs expected resistant founders **L = μ·N₀** (Luria-Delbrück), at 3 levels of **bystander cross-kill** (resistance-agnostic = RUNG-14 ferroptosis_wave/quorum). Validated against the Luria-Delbrück mean; extrapolated analytically to clinical tumour sizes.

**Runtime → set to CPU** (Runtime ▸ Change runtime type ▸ CPU). A GPU would sit idle.

In [ ]:
#@title Cell 1 — clone / pull
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log + VALIDATE (selftest). Stop if it fails.
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = new_log('rung19_escape_race', repo_dir='.')
rc = run_logged([sys.executable,'-u','scripts/45_escape_race.py','selftest'], RUNLOG)
print('[CELL 2]', '✓ validated' if rc==0 else f'✗ FAILED ({rc}) — stop here')

In [ ]:
#@title Cell 3 — RUN the escape race  [CPU, ~1-3 min]
mode = 'run' #@param ['run','quick']
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/45_escape_race.py', mode], RUNLOG)
from IPython.display import Image, display
p='runs/rung19_escape_race/rung19_escape_race'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('HEADLINE:', json.dumps(d['HEADLINE'], indent=2))
    print('\nL* (founders at P(cure)=0.5) by bystander:',
          {k:v['L_star_muN_at_pcure0.5'] for k,v in d['critical_founders_L_star'].items()})
    print('\nclinical extrapolation (no bystander) P(cure):', json.dumps(d['clinical_extrapolation_pcure_no_bystander'], indent=2))

In [ ]:
#@title Cell 4 — bundle zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung19_escape_race_','').replace('.log','')
b = f'/content/rung19_escape_race_{stem}.zip'
ps = sorted(glob.glob('runs/rung19_escape_race/*.png')+glob.glob('runs/rung19_escape_race/*.json')+[str(RUNLOG)])
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 4] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')